In [ ]:
import os
import sys
from google.colab import userdata

# =====================================================================
# 1. SECURE CREDENTIALS & CONSTANTS
# =====================================================================
TOKEN = userdata.get('GithubToken')
USERNAME = "mervynzwkoh"
REPO = "gout-transcriptome-causal"

# Ensure we are operating from the default Colab root directory
%cd /content/

# =====================================================================
# 2. CLONE & INITIALIZE YOUR WORKSPACE
# =====================================================================
print("🗂️ Setting up your personal repository...")
if not os.path.exists(f"/content/{REPO}"):
    !git clone https://{TOKEN}@github.com/{USERNAME}/{REPO}.git
else:
    print("-> Personal repository folder already exists. Skipping clone.")

# Move permanently into your project folder and switch to your feature branch
%cd /content/{REPO}
!git checkout feature/model-fine-tuning
!git pull origin feature/model-fine-tuning

# =====================================================================
# 3. CLONE BIONEMO FRAMEWORK (OUTSIDE YOUR WORKSPACE)
# =====================================================================
print("\n🧬 Fetching external BioNeMo Framework...")
%cd /content/
if not os.path.exists("/content/bionemo-framework"):
    !git clone https://github.com/NVIDIA/bionemo-framework.git
else:
    print("-> BioNeMo framework folder already exists. Skipping clone.")

# =====================================================================
# 4. MODULAR SUB-PACKAGE INSTALLATION
# =====================================================================
print("\n📦 Installing single-cell framework dependencies...")
# Install foundational interfaces and single-cell data loading utilities
!pip install -q -e /content/bionemo-framework/sub-packages/bionemo-core
!pip install -q -e /content/bionemo-framework/sub-packages/bionemo-scdl
!pip install --no-build-isolation transformer_engine[pytorch]
!pip install transformer_engine[jax]

# =====================================================================
# 5. BRIDGE SYSTEM PATH FOR RECIPES
# =====================================================================
# Tell Python's engine exactly where the Geneformer model recipes live
recipe_path = "/content/bionemo-framework/bionemo-recipes"
if recipe_path not in sys.path:
    sys.path.append(recipe_path)

# Return back to your own repository so file saves land in the right spot
%cd /content/{REPO}
print("\n🚀 Setup complete! Workspace and BioNeMo modules are fully synced.")

In [ ]:
import os
import sys
import torch
import torch.nn as nn
from google.colab import drive

# =====================================================================
# 1. MOUNT PERSISTENT STORAGE
# =====================================================================
drive.mount('/content/drive')
weights_path = '/content/drive/MyDrive/Gout_Project_Weights/gout_geneformer_finetuned.pt'

# =====================================================================
# 2. FRAMEWORK PATHING (Assuming BioNeMo is still in /content/)
# =====================================================================
recipe_dir = "/content/bionemo-framework/bionemo-recipes/models/geneformer/src/geneformer"
if recipe_dir not in sys.path:
    sys.path.append(recipe_dir)

from modeling_bert_te import BertModel, TEBertConfig

# =====================================================================
# 3. REBUILD THE ARCHITECTURE
# =====================================================================
model_config = TEBertConfig(
    hidden_size=768, num_attention_heads=12, num_hidden_layers=6,
    vocab_size=40000, max_position_embeddings=2048
)
model_config.position_embedding_type = "absolute"
model_config.is_decoder = False

class GeneformerForRegression(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.bert = BertModel(config)
        self.regression_head = nn.Linear(config.hidden_size, 1)

    def forward(self, input_ids):
        hidden_states = self.bert.embeddings(input_ids=input_ids)
        batch_size, seq_length = input_ids.shape
        attention_mask = torch.zeros((batch_size, 1, 1, seq_length), dtype=hidden_states.dtype, device=hidden_states.device)

        for layer_module in self.bert.encoder.layer:
            layer_outputs = layer_module(hidden_states, attention_mask)
            hidden_states = layer_outputs[0] if isinstance(layer_outputs, tuple) else layer_outputs

        cell_embedding = hidden_states[:, 0, :]
        return self.regression_head(cell_embedding)

# =====================================================================
# 4. REHYDRATE THE BRAIN
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
regression_model = GeneformerForRegression(model_config)

# Load the saved state dictionary from your Google Drive
regression_model.load_state_dict(torch.load(weights_path, map_location=device))
regression_model.to(device)

# Lock the model into Inference Mode
regression_model.eval()

print("🧠 Phase 5 Setup Complete: Fine-tuned model successfully rehydrated and locked for inference.")

In [ ]:
import pickle
import torch

# 1. Configuration
ABCG2_TOKEN_ID = 4803
PAD_TOKEN_ID = 0
MAX_SEQ_LEN = 2048      # Hardware limit

# 2. Load the Baseline Data
with open('data/geneformer_tuning_input.pkl', 'rb') as f:
    data = pickle.load(f)

baseline_sequence = data['tokens'][0]
print(f"🔬 Baseline Cell loaded with {len(baseline_sequence)} expressed genes.")

# 3. Create the Perturbed Sequence safely
if ABCG2_TOKEN_ID in baseline_sequence:
    perturbed_sequence = [PAD_TOKEN_ID if token == ABCG2_TOKEN_ID else token for token in baseline_sequence]
    print("🧬 In-Silico Knockout applied: ABCG2 successfully masked.")
else:
    print("⚠️ ABCG2 not found. Injecting and truncating to maintain memory limits...")
    baseline_sequence.insert(1, ABCG2_TOKEN_ID)

    # TRUNCATION FIX: Slice the array to ensure it never exceeds the MAX_SEQ_LEN
    baseline_sequence = baseline_sequence[:MAX_SEQ_LEN]

    perturbed_sequence = [PAD_TOKEN_ID if token == ABCG2_TOKEN_ID else token for token in baseline_sequence]

# 4. Prepare Tensors
baseline_tensor = torch.tensor([baseline_sequence], dtype=torch.long).to(device)
perturbed_tensor = torch.tensor([perturbed_sequence], dtype=torch.long).to(device)

# 5. Run the Simulation
with torch.no_grad():
    baseline_prediction = regression_model(baseline_tensor).item()
    perturbed_prediction = regression_model(perturbed_tensor).item()

delta_shift = perturbed_prediction - baseline_prediction

print("\n📊 --- EXPERIMENT RESULTS ---")
print(f"Baseline Predicted Dosage State: {baseline_prediction:.4f}")
print(f"Perturbed Predicted Dosage State: {perturbed_prediction:.4f}")
print(f"Δ (Delta) Shift in Cell State: {delta_shift:.4f}")

In [ ]:
import pickle
import torch

ABCG2_TOKEN_ID = 4803
PAD_TOKEN_ID = 0
MAX_SEQ_LEN = 2048

# 1. Load Data
with open('data/geneformer_tuning_input.pkl', 'rb') as f:
    data = pickle.load(f)

print("🔍 Scanning 35 pilot cells for native ABCG2 expression...")

# 2. Search for a valid candidate cell
target_cell_idx = -1
for idx, sequence in enumerate(data['tokens']):
    if ABCG2_TOKEN_ID in sequence:
        target_cell_idx = idx
        break

if target_cell_idx == -1:
    print("❌ Experiment Failed: ABCG2 is not naturally expressed in any of the 35 pilot cells.")
    print("To see a non-zero Delta, you must expand your dataset beyond 35 cells in Phase 3.")
else:
    print(f"✅ Found native ABCG2 in Cell Index {target_cell_idx}!")

    baseline_sequence = data['tokens'][target_cell_idx][:MAX_SEQ_LEN]

    # 3. Apply Knockout
    perturbed_sequence = [PAD_TOKEN_ID if token == ABCG2_TOKEN_ID else token for token in baseline_sequence]

    baseline_tensor = torch.tensor([baseline_sequence], dtype=torch.long).to(device)
    perturbed_tensor = torch.tensor([perturbed_sequence], dtype=torch.long).to(device)

    # 4. Inference
    with torch.no_grad():
        baseline_prediction = regression_model(baseline_tensor).item()
        perturbed_prediction = regression_model(perturbed_tensor).item()

    delta_shift = perturbed_prediction - baseline_prediction

    print("\n📊 --- NATIVE EXPERIMENT RESULTS ---")
    print(f"Baseline Predicted Dosage State: {baseline_prediction:.4f}")
    print(f"Perturbed Predicted Dosage State: {perturbed_prediction:.4f}")
    print(f"Δ (Delta) Shift in Cell State: {delta_shift:.4f}")

In [ ]:
import pickle
import urllib.request
import os

# 1. Ensure the dictionary exists locally; if not, download it from Hugging Face
dict_path = "token_dictionary_gc104M.pkl"
url = "https://huggingface.co/ctheodoris/Geneformer/resolve/main/geneformer/token_dictionary_gc104M.pkl"

if not os.path.exists(dict_path):
    print("📥 Downloading official Geneformer token dictionary...")
    urllib.request.urlretrieve(url, dict_path)

# 2. Load the Dictionary mapping (Ensembl ID -> Token ID)
with open(dict_path, "rb") as f:
    gene_token_dict = pickle.load(f)

# 3. Define the Ensembl IDs for your Gout targets
target_genes = {
    "ABCG2": "ENSG00000118777",
    "SLC22A12 (URAT1)": "ENSG00000198467"
}

# 4. Programmatically look up their exact integer Token IDs
print("\n🔍 --- OFFICIAL TOKEN ID LOOKUP ---")
for gene_name, ensembl_id in target_genes.items():
    token_id = gene_token_dict.get(ensembl_id)
    if token_id:
        print(f"✅ {gene_name} ({ensembl_id}) strictly maps to Token ID: {token_id}")
    else:
        print(f"❌ {gene_name} not found in the pre-trained dictionary.")